# 04 - Optimize

**Where this fits:** the metamodel trained in `03_train_metamodel.ipynb` is a
fast stand-in for the real (slow, expensive) microsimulation. This notebook is
the payoff for having built it — since the metamodel is just a mathematical
function, we can now search over millions of possible kit allocations in
minutes instead of running the actual simulation millions of times, which
would be computationally infeasible.

**In plain terms:** think of the metamodel as a landscape with hills and
valleys, where the height at any point represents predicted overdose deaths
for that allocation. We want the lowest valley. A single optimization run can
get stuck in a *local* valley that isn't the deepest one overall, so this
notebook starts the search from 1,000 different, spread-out points across
that landscape and lets each one roll downhill independently
(`src/optimization.py`). Whichever of the 1,000 resulting valleys is deepest
is our best candidate — that's what `05_find_optimal_solutions.ipynb` picks up
next.

## Overview
Using the trained metamodel from `03_train_metamodel.ipynb` as a surrogate 
objective function, this notebook runs a numerical optimization to identify 
naloxone kit distributions that minimize projected overdose deaths across 
Rhode Island cities and towns.

## Inputs
- `models/model_weights.pkl` — trained metamodel weights (not included in this public repo — see root README)
- `models/scaler.pkl` — feature scaler (same as above)

## Output
- `results/outputs/Local_Minima.csv` — 1,000 local minima naloxone distributions 
with projected death counts as suggested by the metamodel

## Workflow
1. Generate random starting points across the solution space (`x0_list.npy`)
2. Run gradient-based optimization from each starting point (`src/optimization.py`)
3. Combine local minima results into a single output file

> **Note:** Step 2 was executed on a university HPC cluster using SLURM job 
> scheduling due to the computational demands of running 1,000 optimization 
> instances in parallel. The SLURM submission script itself isn't included in
> this public repo (it's just cluster-specific job scheduling boilerplate),
> but `src/optimization.py` — the actual optimization logic each job ran — is
> included in full.</cell id="349bcd5e">

## Generate Starting Points

This step generates the starting points used for the local optimization runs
(1,000 diverse initial allocation vectors), each summing to the fixed statewide
budget of 50,000 kits.

**Input:**

- Rhode Island Demographics (Rhode_Island_Demographic.xlsx)

**Output:** 

- Starting-point samples for the local optimization process (x0_list.npy)

In [7]:
# Import Libraries

import pandas as pd
from pathlib import Path

import numpy as np

In [5]:
# Repo root (1 levels up from notebooks/)
BASE_DIR = Path.cwd().parents[0]

# Load population proportions computed in 01_generate_samples.ipynb
proportions = pd.read_csv(BASE_DIR / 'data' / 'generated' / 'population_proportions.csv', 
                          index_col=0).squeeze("columns")

In [10]:
from scipy.stats import qmc

# Generate integer-valued naloxone allocation vectors using Sobol sampling
# biased by population weights, each summing to exactly 50,000 kits

DIM = 39
N_SAMPLES = 1000
TOTAL_KITS = 50000
SEED = 42

# Normalize population weights
p = proportions.to_numpy()

# Generate Sobol samples biased by population weights
sampler = qmc.Sobol(d=DIM, scramble=True, seed=SEED)
samples = sampler.random(N_SAMPLES)

# Bias by population weights and renormalize
x_norm = samples * p
x_norm /= x_norm.sum(axis=1, keepdims=True)

# Scale to total kit count
X = TOTAL_KITS * x_norm

# Convert to integers while preserving exact row sums
X_int = np.floor(X).astype(int)
remainders = TOTAL_KITS - X_int.sum(axis=1)

# Distribute remaining kits to cities with largest fractional parts
frac_parts = X - X_int
for i in range(N_SAMPLES):
    if remainders[i] > 0:
        top_indices = np.argsort(frac_parts[i])[::-1]
        X_int[i, top_indices[:remainders[i]]] += 1

print("Example allocation vector:", X_int[0])
print("Total kits allocated:", X_int[0].sum())


Example allocation vector: [ 597 1710 1125   84  209 2584 4968 1768  913  479  305  275  330  426
  207  934  142   84 1076 1447   25  545 1543 1814   91 3405  949 9324
  163  573 1171 1187  884   47 2916  306  976 1693 2725]
Total kits allocated: 50000


/var/folders/p3/jblj8thn72vcyjg4m0sm_t680000gn/T/ipykernel_58434/3664967715.py:16: UserWarning: The balance properties of Sobol' points require n to be a power of 2.
  samples = sampler.random(N_SAMPLES)


In [11]:
row_sums = X_int.sum(axis=1)
print(row_sums)

[50000 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000
 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000
 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000
 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000
 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000
 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000
 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000
 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000
 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000
 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000
 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000
 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000
 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000 50000
 50000 50000 50000 50000 50000 50000 50000 50000 50

In [12]:
# Save optimization starting points for use in HPC optimization runs
(BASE_DIR / 'data' / 'generated').mkdir(parents=True, exist_ok=True)
np.save(BASE_DIR / 'data' / 'generated' / 'x0_list.npy', X_int)

## Run Local Optimizations

Each of the 1,000 starting points above was optimized independently using
`src/optimization.py`, executed in parallel on a SLURM-based HPC cluster. Each
run produces one local minimum: an allocation vector and its projected death
count. The cell below combines the resulting batch files into a single
dataset of 1,000 local minima.</cell id="b1d83575">

In [ ]:
# Combine optimization results from HPC cluster runs into a single DataFrame
# NOTE: This code was executed on a university HPC cluster and is provided
# for documentation purposes only. File paths are environment-specific.

from pathlib import Path
import pandas as pd

N_BATCHES = 100

CITIES = [
    "Barrington", "Bristol", "Burrillville", "Central Falls", "Charlestown", "Coventry",
    "Cranston", "Cumberland", "East Greenwich", "East Providence", "Exeter", "Foster",
    "Glocester", "Hopkinton", "Jamestown", "Johnston", "Lincoln", "Little Compton",
    "Middletown", "Narragansett", "New Shoreham", "Newport", "North Kingstown", "North Providence",
    "North Smithfield", "Pawtucket", "Portsmouth", "Providence", "Richmond", "Scituate",
    "Smithfield", "South Kingstown", "Tiverton", "Warren", "Warwick", "West Greenwich",
    "West Warwick", "Westerly", "Woonsocket"
]

HPC_PATH = Path("/users/1/kuntz138/Python_Optimization")

# Load and concatenate all batch results
final_df = pd.concat(
    [pd.read_csv(HPC_PATH / f"result_batch_{i}.csv", header=None) 
     for i in range(1, N_BATCHES + 1)],
    ignore_index=True
)

final_df.columns = ['Minimized_Value'] + CITIES

print(final_df.shape)

In [ ]:
# Save local minima results
(BASE_DIR / 'results' / 'outputs').mkdir(parents=True, exist_ok=True)
final_df.to_csv(BASE_DIR / 'results' / 'outputs' / 'Local_Minima.csv', index=False)